# Lab 0: Welcome to Qiskit Fall Fest 2026 Warsaw

Welcome! This is Lab 0, the warm-up notebook for the Qiskit Fall Fest 2026 Warsaw labs. In this lab, you will:

- Upgrade your Python packages
- Complete simple coding exercises with Sampler and Estimator
- Build a special multi-qubit quantum circuit and explore its statevector properly

Excited? There's plenty more ahead!


This lab runs on macOS, Linux, and cloud platforms like [qBraid](https://qbraid.com/) and [Google Colab](https://colab.research.google.com/). If you're on Windows, we recommend using a cloud platform for the best experience, especially for the upcoming labs where some packages work best in Linux-based environments.

Here's a quick summary:

| Your OS | How to run this lab |
|---|---|
| macOS / Linux | Run locally — the notebook handles setup for you. |
| Windows | Run locally with Python and venv, or open this notebook on [qBraid](https://qbraid.com/) or [Google Colab](https://colab.research.google.com/). |

<br>


## Installation

Run this cell first to check your Python version.

In [4]:
# Install the packages this lab needs. Run once per environment, then restart the kernel.
# Qiskit Fall Fest 2026 Warsaw — labs run on any Python 3.11+ environment.
%pip install -q \
    "qiskit[visualization]" \
    "qiskit-aer" \
    "qiskit-ibm-runtime" \
    "matplotlib" \
    "setuptools" \
    "qfest-grader"\
    "dotenv"


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
import sys

recommended = (3, 10)
current = sys.version_info[:2]

if current < recommended:
    print(f"You are using Python {current[0]}.{current[1]}. Python {recommended[0]}.{recommended[1]} or later is recommended.")
else:
    print(f"Python {current[0]}.{current[1]} — looks good!")


Python 3.13 — looks good!


Run the install cell above once per environment to get Qiskit, qiskit-aer, qiskit-ibm-runtime, matplotlib, and qfest-grader. Then continue to the next section.

In [2]:
import os
import platform
import shutil
import subprocess
import sys

import qfest_grader
import qiskit

from qfest_grader.challenges.warsaw_2026 import check_progress
from qfest_grader.challenges.warsaw_2026.lab0 import (
    grade_lab0_ex1,
    grade_lab0_ex2,
    grade_lab0_ex3,
    grade_lab0_ex4,
)


from qiskit.circuit import QuantumCircuit
from qiskit.quantum_info import SparsePauliOp, Statevector
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
from qiskit.visualization import array_to_latex, plot_histogram, plot_state_qsphere
from qiskit_aer import AerSimulator
from qiskit_ibm_runtime import Batch, QiskitRuntimeService
from qiskit_ibm_runtime import EstimatorV2 as Estimator
from qiskit_ibm_runtime import SamplerV2 as Sampler

import matplotlib.pyplot as plt

print(f"Qiskit version: {qiskit.__version__}")
print(f"Grader version: {qfest_grader.__version__}")



Qiskit version: 2.6.0
Grader version: 0.3.0


# Chapter 1 — Your first quantum program with Qiskit patterns

## 1.1 IBM Quantum Account

First things first! To run these labs, you'll need an active IBM Quantum account.

If you've already saved your account credentials with `QiskitRuntimeService`, you can just run the cell below and your backend list will appear. If you're a new user, or it's been a while since you last set things up, follow these steps to save your account:

1. Go to [quantum.cloud.ibm.com](https://quantum.cloud.ibm.com/) and sign in (or create an account).
2. Copy your API key by clicking `Create +` or `View all` on the dashboard.
3. Copy your instance CRN from the Instances tab.
4. Replace the placeholders in the cell below with your API key and Instance CRN, then run it.

Once that's done, save your grader email in the cell after it, then run the Exercise 1 grader cell to confirm everything is set up.

In [3]:
# Paste your IBM Cloud credentials here. You only need to do this once per environment.
# After it works, you can comment these lines out so the credentials do not stay in the file.

#token = "<YOUR_API_KEY>"
#instance = "<YOUR_CRN>"  # looks like "crn:v1:bluemix:public:quantum-computing:..."

from dotenv import load_dotenv

# 1. Load the variables from the .env file into the system environment
load_dotenv()

# 2. Retrieve the variables
my_api_key = os.getenv("IBM_QUANTUM_API_KEY")
my_crn = os.getenv("IBM_QUANTUM_CRN")


QiskitRuntimeService.save_account(
    token=my_api_key,
    instance=my_crn,
    overwrite=True,
    set_as_default=True,
    
)

# Verify
service = QiskitRuntimeService()
backends = service.backends()
print(f"Account OK. {len(backends)} backend(s) available:")
for b in backends[:5]:
    print(f"  {b.name} ({b.num_qubits} qubits)")


Account OK. 3 backend(s) available:
  ibm_fez (156 qubits)
  ibm_kingston (156 qubits)
  ibm_marrakesh (156 qubits)


In [4]:
# Grader account: save the email you registered with for Qiskit Fall Fest 2026 Warsaw.
# The grader records your completed exercises under this email (participation certificates use these records).
# Run once per environment; a new Colab session needs it again.
from qfest_grader import save_account

save_account("rjinsta102@gmail.com")

Grader account saved for rjinsta102@gmail.com. ✅


<div class="alert alert-block alert-success">
  <b>Exercise 1: Verify your grader account</b>

  Have you set your API key and CRN, checked the available backends and saved your grader email?
  Run the Exercise 1 cell below to make sure the grader works properly.
  You don’t need to enter anything!

  If your grader account is saved correctly, you will see a success message.
</div>


In [5]:
grade_lab0_ex1()

Grading your answer. Please wait...

Congratulations! 🎉 Your grader account works. Your answers are recorded for rjinsta102@gmail.com.
You scored 1.0 on this exercise.


## 1.2 Quantum Computing with Gates

In Qiskit, we build quantum circuits using [`QuantumCircuit`](https://quantum.cloud.ibm.com/docs/api/qiskit/qiskit.circuit.QuantumCircuit) and apply gates with circuit methods.

The [Hadamard gate (H)](https://quantum.cloud.ibm.com/docs/api/qiskit/qiskit.circuit.library.HGate) maps $|0\rangle$ to the equal superposition state $H|0\rangle = |+\rangle = \frac{1}{\sqrt{2}}\bigl(|0\rangle + |1\rangle\bigr)$. When you measure $H|0\rangle$, you get 0 or 1 with equal probability (50% each).

Complete the cell below to build your first 1-qubit circuit with a Hadamard gate.


<div class="alert alert-block alert-success">
<b>Exercise 2: Build your first quantum circuit</b>

Below is an empty 1-qubit circuit. Add a <b>Hadamard gate</b> on qubit 0 to put it into the superposition $H|0\rangle = |+\rangle$. This gives outcome probabilities of 50/50, so the Sampler in Section 1.6 returns roughly equal counts (generally not exactly equal, because the number of shots is finite).

Hint: try our official [QuickStart](https://quantum.cloud.ibm.com/docs/guides/quick-start).

</div>


In [6]:

qc = QuantumCircuit(1)

# TODO: Add a Hadamard gate on qubit 0
qc.h(0)
qc.draw()
#qc.draw('mpl')

┌───┐
q: ┤ H ├
   └───┘

## 1.3 What is a Qiskit pattern?

We now have a quantum circuit that prepares the $H|0\rangle$ superposition state. In the next sections, we will run two experiments on this circuit: one to collect measurement counts, and one to compute the expectation value of the Z observable. Both experiments follow the same four-step workflow called a [Qiskit pattern](https://quantum.cloud.ibm.com/docs/guides/intro-to-patterns).

| Step | What you do | In this lab |
|------|------------|-------------|
| 1. Map | Map the problem to quantum circuits and operators. | We build the H circuit and define the Z observable. |
| 2. Optimize | Optimize the circuit for target hardware. | We use `generate_preset_pass_manager` to do this. |
| 3. Execute | Execute on target hardware using primitives. | We run the circuit with both Sampler and Estimator. |
| 4. Post-process | Post-process results. | We plot the measurement histogram and verify that ⟨Z⟩ ≈ 0. |

For the Execute step, Qiskit provides two primitives:

- [SamplerV2](https://quantum.cloud.ibm.com/docs/guides/primitives) samples the output register from quantum circuit execution. We use it to collect measurement counts and verify that 0 and 1 appear with roughly equal frequency.
- [EstimatorV2](https://quantum.cloud.ibm.com/docs/guides/primitives) estimates expectation values of observables with respect to states prepared by quantum circuits, and reports a standard error for each estimate. We use it to evaluate $\langle Z \rangle$, where $|0\rangle$ has eigenvalue $+1$ and $|1\rangle$ has eigenvalue $-1$, giving $\langle Z \rangle = 0$ for an equal superposition.

The next sections go through each step.

## 1.4 Step 1 — Map

In the Map step, we define the quantum problem as a circuit and an observable.

The circuit is already built: `qc` prepares the $H|0\rangle$ state. Here we define the observable. We use the Z operator (Pauli-Z), which has eigenvalue $+1$ for $|0\rangle$ and $-1$ for $|1\rangle$:

$$Z = |0\rangle\langle 0| - |1\rangle\langle 1|$$

For the $H|0\rangle$ state, both outcomes are equally likely, so the expectation value is:

$$\langle Z \rangle = \frac{1}{2}(+1) + \frac{1}{2}(-1) = 0$$

This is what we will verify with Estimator in the Execute step.

<div class="alert alert-block alert-success">
<b>Exercise 3: Define the observable</b>

In the Map step, you choose what to measure. We want the Pauli-Z operator on qubit 0; its expectation value tells us how biased the qubit is toward |0⟩ vs |1⟩.

Create a single-qubit Z __observable__ using <code>SparsePauliOp</code> with a coefficient $1.0$.

Hint: you can find nice examples in our [SparsePauliOp documentation](https://quantum.cloud.ibm.com/docs/api/qiskit/qiskit.quantum_info.SparsePauliOp).

</div>


In [ ]:

# TODO: Define the observable as the single-qubit Z operator
observable = None

print(f"Observable: {observable}")


## 1.5 Step 2 — Transpile

The `Transpile` step rewrites your abstract circuit into an equivalent circuit that the target backend can execute. We use `generate_preset_pass_manager` to handle this.

Transpilation applies to both the circuit and the observable. When the transpiler assigns virtual qubits to physical qubits, the observable must reflect that same mapping. We do this with `apply_layout`. Since our circuit has only one qubit, the layout is trivial here, but the same pattern is required for multi-qubit circuits on real hardware.


In [ ]:
backend = AerSimulator()
pm = generate_preset_pass_manager(optimization_level=1, backend=backend)

# Sampler requires measurements in the circuit
qc_meas = qc.copy()
qc_meas.measure_all()
isa_qc_meas = pm.run(qc_meas)

# Estimator takes the circuit without measurements and evaluates the observable separately
isa_qc = pm.run(qc)

# Apply the same layout to the observable so qubit indices match the transpiled circuit
isa_observable = observable.apply_layout(isa_qc.layout)

print("Transpiled circuit and observable ready.")

## 1.6 Step 3 — Execute

Now we run the transpiled circuit using both primitives. Each answers a different question about the same quantum state.

### Sampler: "What counts do I get?"

The Sampler runs your circuit many times (shots) and counts how often each outcome occurs: how many times do I measure 0, and how many times do I measure 1?

In [ ]:
sampler = Sampler(mode=backend)
sampler_result = sampler.run([isa_qc_meas], shots=1000).result()
counts = sampler_result[0].data.meas.get_counts()

print("Sampler counts:", counts)

<div class="alert alert-block alert-success">
<b>Submit & check your Sampler result to complete Exercise 2</b>

The <code>counts</code> you just computed comes from the H circuit you built in Section 1.2. Run the cell below to submit it; the grader accepts counts close to 50/50, allowing for shot noise.

</div>


In [ ]:
grade_lab0_ex2(counts)

### Estimator: "What value do I get when I observe my quantum circuit?"

Estimator takes the circuit and the observable and returns an estimate of the expectation value, with its standard error (`stds`), without you having to collect and average counts manually. The circuit does not need measurement gates — Estimator handles the measurement internally.

For the $H|0\rangle$ state the exact value is $\langle Z \rangle = 0$. The estimate will typically differ from 0 by about ±0.016, because the Estimator uses 4096 shots by default, even on the simulator.


In [ ]:
# Run Estimator — circuit WITHOUT measurements + observable
estimator = Estimator(mode=backend)
estimator_result = estimator.run([(isa_qc, isa_observable)]).result()
exp_val = estimator_result[0].data.evs

print(f"⟨Z⟩ = {exp_val:.4f}")

<div class="alert alert-block alert-success">
<b>Submit & check your Estimator result to complete Exercise 3</b>

Run the cell below to submit <code>exp_val</code> to the grader. For a perfect H|0⟩ superposition, ⟨Z⟩ should be ≈ 0.

</div>


In [ ]:
grade_lab0_ex3(exp_val)

## 1.7 Step 4 — Post-process

The Post-process step is where we extract meaning from the raw results. Depending on the problem, this can mean visualizing measurement distributions, computing derived quantities from expectation values, feeding results into a classical optimization loop, or applying error mitigation techniques.

In this lab, we keep it simple. We plot the Sampler counts as a histogram to see how often 0 and 1 were measured, and we plot the Estimator result as a bar chart with an error bar showing the statistical uncertainty on $\langle Z \rangle$. On the ideal simulator we expect counts close to 500/500 and $\langle Z \rangle$ close to 0, within shot noise (about ±16 counts for 1000 shots, and about ±0.016 for the Estimator, which uses 4096 shots by default). On real hardware, device noise adds to this shot noise.

In [ ]:

# Sampler results
print(f"Sampler counts: {counts}")
display(plot_histogram(counts, title="H|0⟩ measurement distribution (1000 shots)"))

# Estimator results
ev = estimator_result[0].data.evs.item()
std = estimator_result[0].data.stds.item()
print(f"\nEstimator ⟨Z⟩ = {ev:.4f} ± {std:.4f}")

fig, ax = plt.subplots(figsize=(4, 5))
ax.bar(["⟨Z⟩"], [ev], yerr=[std], capsize=10, color="steelblue", width=0.4)
ax.axhline(0, color="gray", linewidth=0.8, linestyle="--")
ax.set_ylim(-1.2, 1.2)
ax.set_ylabel("Expectation value")
ax.set_title("Z observable on H|0⟩ (Estimator)")
plt.tight_layout()
plt.show()


## 1.8 (Optional) Run on Real Quantum Hardware

<div class="alert alert-block alert-info">
<b>This section is optional and ungraded.</b> It submits jobs to a real quantum computer and uses your QPU time. You can skip it and continue to Chapter 2.
</div>

So far we used a simulator. Now we run the same circuit on a real IBM quantum processor. The code follows the same Qiskit Patterns workflow. The only change is the backend.

Real quantum hardware operates under physical constraints that simulators do not have. Gates run as microwave pulses on physical qubits, and each operation has a finite fidelity. Hardware noise can therefore shift the counts away from 50/50 and $\langle Z \rangle$ away from 0, on top of the shot-to-shot fluctuations that the simulator also shows. On hardware the Estimator applies readout-error mitigation by default (`resilience_level=1`), while the Sampler returns raw counts, so keep that in mind when you compare them.

We submit both Sampler and Estimator jobs together using [Batch](https://quantum.cloud.ibm.com/docs/guides/run-jobs-batch). Batch groups multiple jobs into a single backend reservation so they run back-to-back without re-queuing. This job uses approximately 20 seconds of QPU time with `Heron` devices. Queue wait time varies depending on backend load. If the cell times out before printing the job IDs, the following cell shows how to retrieve the results by job ID.

In [ ]:
# By default, this section runs on real quantum hardware.
# To save your QPU time for the later labs, uncomment the noisy simulator line below
# and replace real_backend with real_backend_sim in the rest of this section.

real_backend = service.least_busy()
# real_backend_sim = AerSimulator.from_backend(real_backend)  # noisy simulator

print(f"Running on: {real_backend.name} ({real_backend.num_qubits} qubits)")

# Step 2: Transpile for the target backend
pm_real = generate_preset_pass_manager(optimization_level=1, backend=real_backend)

qc_meas = qc.copy()
qc_meas.measure_all()
isa_qc_meas = pm_real.run(qc_meas)

isa_qc_real = pm_real.run(qc)
isa_obs = observable.apply_layout(isa_qc_real.layout)

# Step 3: Execute — both primitives run in a single Batch to avoid re-queuing
with Batch(backend=real_backend):
    sampler = Sampler()
    sampler.options.environment.job_tags = ["fallfest26-warsaw"]
    sampler_job = sampler.run([isa_qc_meas], shots=1000)

    estimator = Estimator()
    estimator.options.environment.job_tags = ["fallfest26-warsaw"]
    estimator_job = estimator.run([(isa_qc_real, isa_obs)])

sampler_job_id = sampler_job.job_id()
estimator_job_id = estimator_job.job_id()

print(f"Sampler job ID:   {sampler_job_id}")
print(f"Estimator job ID: {estimator_job_id}")


In [ ]:
# (Optional) If the cell above timed out before printing the results, copy the
# Sampler/Estimator job IDs from its output into the strings below and run this cell.
# Leave commented otherwise.

# sampler_job_id = "<paste sampler job id>"
# estimator_job_id = "<paste estimator job id>"
# sampler_job = service.job(sampler_job_id)
# estimator_job = service.job(estimator_job_id)


In [ ]:
# Step 4: Results
hw_counts = sampler_job.result()[0].data.meas.get_counts()
hw_estimator_result = estimator_job.result()
hw_exp_val = hw_estimator_result[0].data.evs

print(f"Hardware Sampler counts: {hw_counts}")
print(f"Hardware Estimator ⟨Z⟩:  {hw_exp_val:.4f}")

Compare the results side by side. For an ideal circuit the expected outcome probabilities are 50/50. Both runs use a finite number of shots, so the counts are generally not exactly 500/500 (an ideal 1000-shot run gives exactly 500/500 only about 2.5% of the time): the count of `0` fluctuates by about ±16 (one standard deviation, $\sqrt{1000\cdot\tfrac12\cdot\tfrac12}\approx 15.8$). A hardware deviation from 500 is evidence of device noise only when it is clearly larger than this spread (roughly ±31 counts at 95% confidence). Real hardware can add such deviations through gate and measurement errors; readout errors are often asymmetric, which tends to bias the counts toward `0`. The cell below also prints each deviation in units of this standard deviation.

In [ ]:
display(plot_histogram(
    [counts, hw_counts],
    legend=["Simulator", f"Hardware ({real_backend.name})"],
    title="H|0⟩ — Simulator vs Real Hardware",
))


# Estimator results
ev = hw_estimator_result[0].data.evs.item()
std = hw_estimator_result[0].data.stds.item()
print(f"\nEstimator ⟨Z⟩ = {ev:.4f} ± {std:.4f}")

fig, ax = plt.subplots(figsize=(4, 5))
ax.bar(["⟨Z⟩"], [ev], yerr=[std], capsize=10, color="steelblue", width=0.4)
ax.axhline(0, color="gray", linewidth=0.8, linestyle="--")
ax.set_ylim(-1.2, 1.2)
ax.set_ylabel(f"Expectation value from ({real_backend.name})")
ax.set_title("Z observable on H|0⟩ (Estimator)")
plt.tight_layout()
plt.show()

# Shot-noise check: distance of P(0) from 0.5 in units of the binomial standard deviation
import numpy as np


def p0_and_z(c):
    shots = sum(c.values())
    p0 = c.get("0", 0) / shots
    sigma = np.sqrt(0.25 / shots)  # 1σ binomial shot noise if the true p0 is 1/2
    return p0, (p0 - 0.5) / sigma


for label, c in [("Simulator", counts), (f"Hardware ({real_backend.name})", hw_counts)]:
    p0, z = p0_and_z(c)
    print(f"{label}: P(0) = {p0:.3f}, deviation from 0.5 = {z:+.1f} σ")
print("|z| < 2 fits shot noise alone. |z| > 2 hints at device noise, but an ideal simulator also exceeds 2 in about 1 run in 20.")


# Chapter 2 — Multi-qubit circuit and visualization

Great work on Chapter 1! You built a single-qubit superposition, ran it through Sampler and Estimator, and saw how the same quantum state looks from two different measurement perspectives — on both a simulator and, if you tried it, real quantum hardware.

Now let's build something more interesting. In this chapter, you will work with 6 qubits to construct a special quantum circuit step by step. At the end, we will look at the statevector it produces and visualize it using [QSphere](https://quantum.cloud.ibm.com/docs/api/qiskit/qiskit.visualization.plot_state_qsphere), a multi-qubit visualization tool that shows the probability and phase of each basis state. You might recognize what the final state looks like.

## 2.1 Build a 6-qubit entangled state

You will use three gates in this exercise:

- [H](https://quantum.cloud.ibm.com/docs/api/qiskit/qiskit.circuit.library.HGate): `qc.h(qubit)` applies a Hadamard gate to `qubit`.
- [X](https://quantum.cloud.ibm.com/docs/api/qiskit/qiskit.circuit.library.XGate): `qc.x(qubit)` flips `qubit` between $|0\rangle$ and $|1\rangle$.
- [CX](https://quantum.cloud.ibm.com/docs/api/qiskit/qiskit.circuit.library.CXGate): `qc.cx(control, target)` applies a two-qubit CNOT gate from `control` to `target`.

All qubits start in $|0\rangle$. Start with the cell below.


<div class="alert alert-block alert-success">
<b>Exercise 4: Build a 6-qubit entangled state</b>

You will build a 6-qubit circuit. When you visualize it on a QSphere later, you will see something familiar.

Build it in three steps using the cells below:

1. (already given) Create a 6-qubit circuit.
2. Apply <b>H</b> to qubit 0 to put it into the superposition $H|0\rangle = |+\rangle$, then apply <b>X</b> to qubit 1.
3. Apply a <b>CNOT chain</b> with qubit 0 as control and qubits 1 through 5 as targets to entangle them all.

Hint: you can use a `for` loop to build the CNOT chain.
</div>

In [ ]:
# Step 1: Create a 6-qubit circuit
special_qc = QuantumCircuit(6)
special_qc.draw()
#special_qc.draw('mpl') 

Next, apply H to qubit 0 and X to qubit 1.

In [ ]:
# Step 2: H on qubit 0, X on qubit 1
# TODO: Apply H to qubit 0

# TODO: Apply X to qubit 1


special_qc.draw()
#special_qc.draw('mpl') 


Now apply a CNOT chain where each gate uses qubit 0 as the control and targets qubits 1 through 5.


In [ ]:
# Step 3: CNOT chain — entangle all qubits with qubit 0
# TODO: Loop over qubits 1 through 5 and apply CNOT with qubit 0 as control


special_qc.draw()
#special_qc.draw('mpl') 


## 2.2 Statevector

For $n$ qubits, a statevector is a complex vector of $2^n$ amplitudes $\alpha_k$, one for each possible basis state $|k\rangle$, with measurement probability $p_k = |\alpha_k|^2$.

For our 6-qubit circuit, the statevector has $2^6 = 64$ entries, covering every basis state from $|000000\rangle$ to $|111111\rangle$. The `Statevector` class computes this from the circuit, and `array_to_latex` prints it in mathematical form.

Most entries will be zero. Only the basis states with non-zero amplitude contribute to the quantum state you built.

In [ ]:
sv = Statevector(special_qc)
array_to_latex(sv, max_size=64)

## 2.3 Visualize your quantum state

Reading 64 complex numbers directly is not practical. The [QSphere](https://quantum.cloud.ibm.com/docs/api/qiskit/qiskit.visualization.plot_state_qsphere) puts a pure multi-qubit state on a single sphere so we can see its structure at a glance. Each basis state is a point on the sphere, with three properties:

- Latitude is determined by the Hamming weight (the number of 1s in the bitstring). States with no 1s sit at the north pole; states with all 1s sit at the south pole.
- Dot size represents the measurement probability of that basis state. States with zero amplitude do not appear.
- Color represents the quantum phase.

For the state you built, you will see two dots of equal size. Take a look at which basis states they correspond to.

In [ ]:
# The QSphere
plot_state_qsphere(sv)

### What the QSphere shows

The QSphere has two dots, one near the top and one near the bottom. They are the two basis states in equal superposition, linked by entanglement.

<div class="alert alert-block alert-success">
<b>Exercise 4 — Submit your statevector</b>

Submit your <code>sv</code> to the grader to complete Exercise 4. The grader will check your circuit and reveal why this state is special. You will also receive links to courses that will help you prepare for the later labs. Run the cell below to wrap up Lab 0.

</div>


In [ ]:
grade_lab0_ex4(sv)


### What is next

IBM put its first quantum computer on the cloud on May 4, 2016. Since then, Qiskit and the quantum computing community have grown together through each generation of hardware, software, and education. Qiskit is built by its community, and it will be with you throughout your quantum journey.

In this lab, you ran a quantum circuit on a simulator and on real hardware, measured a quantum state with two different primitives, and visualized a 6-qubit entangled state. These are the core building blocks you will use throughout Qiskit Fall Fest 2026 Warsaw.

Let's wrap up the main exercises by checking your progress.


In [ ]:
check_progress("lab0")